# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [76]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

#if CURRENT_DIR.name == 'notebooks':
if CURRENT_DIR.name == 'chapter05':  #내 과제에서는 llm-data-analysis-course\practice\chapter05 폴더 안에서 실행
    PROJECT_ROOT = CURRENT_DIR.parent.parent # 2단계 상위 폴더로 이동하여 프로젝트 루트 경로를 설정
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT /'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: d:\dev\llm-data-analysis-course\practice\chapter05
프로젝트 루트: d:\dev\llm-data-analysis-course
원본 데이터 폴더: d:\dev\llm-data-analysis-course\data\raw
전처리 데이터 폴더: d:\dev\llm-data-analysis-course\data\processed
보고서 폴더: d:\dev\llm-data-analysis-course\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [77]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


### [💫 Notes 💫] - 데스크탑 변경 예정이라 기록해둠!
---
##### 오류 해결
1. 경로가 이상할때 -> Notebook은 저장소 내부에서 실행할 것!
```python
from pathlib import Path
print(Path.cwd())
```

2. 파일이 없을 때.

```powershell
python scripts/generate_sample_data.py
```

## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [78]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [79]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [80]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [81]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum()) #True를 1, False를 0으로 계산하여 컬럼별 결측치 개수 집계



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [82]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2), # 각 컬럼의 결측치 비율을 소수점 둘째 자리까지 반올림하여 백분율 계산
    })
    return summary.sort_values('missing_count', ascending=False)#내림차순

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [83]:
if 'age' in customers_clean.columns: #age 컬럼이 존재하는지 확인
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')  # age 컬럼을 숫자형 데이터로 변환, 숫자로 변환할 수 없는 값은 NaN으로 처리
    age_median = customers_clean['age'].median() # age 컬럼에서 결측치를 제외한 값들의 중앙값 계산
    customers_clean['age'] = customers_clean['age'].fillna(age_median) # fillna(): 결측치를 지정한 값으로 채우는 함수
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')  # city 컬럼의 결측치를 'Unknown' 문자열로 대체

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [84]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum()) #True의 개수를 합산하여 중복 행 개수 계산


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [85]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [86]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


### [💫 Notes 💫]

- isna(): 현재 값이 결측치인지 확인
- astype(str): 값을 문자열로 변환
- str.strip(): 문자열의 앞뒤 공백 및 줄바꿈 등을 제거
- where()의 조건이 False인 위치에 해당 결과를 사용
- strip_string_columns 문자열 컬럼에서 앞뒤 공백 제거

In [87]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),# True이면 원래 결측치를 그대로 유지
            result[col].astype(str).str.strip()
        ) #where(조건, 대체값)은 조건이 참이면 기존 값을 유지하고, 거짓이면 대체값을 사용

    return result

customers_clean = strip_string_columns(customers_clean) 
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\KimHyeBin\AppData\Local\Temp\ipykernel_33596\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)
C:\Users\KimHyeBin\AppData\Local\Temp\ipykernel_33596\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and s

In [88]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [89]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [90]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-09-15 00:00:00
주문 종료일: 2026-09-14 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-14,2026-07,Tuesday
1,2025-09-29,2025-09,Monday
2,2026-01-26,2026-01,Monday
3,2026-04-08,2026-04,Wednesday
4,2026-02-27,2026-02,Friday


In [91]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-08-26
1,2,2026-01-09
2,3,2024-08-19
3,4,2026-06-20
4,5,2024-11-20


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [92]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [93]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [94]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [95]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [96]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [97]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [98]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [99]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [100]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('d:/dev/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('d:/dev/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('d:/dev/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('d:/dev/llm-data-analysis-course/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [101]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: d:\dev\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [102]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        # to_numeric(): 숫자형 변환, errors='coerce': 변환 실패 시 NaN
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        # fillna(): 결측치 대체, median(): 중앙값 계산
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        # fillna(): 결측치를 'Unknown'으로 대체
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        # to_datetime(): 날짜형 변환, 변환 실패 시 NaT
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        # replace(): 지정한 값으로 치환
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        # to_datetime(): 날짜형 변환
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        # dt.to_period('M'): 월 단위 변환, astype(str): 문자열 변환
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        # dt.day_name(): 날짜에서 요일 이름 추출
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    # issubset(): 지정한 컬럼이 모두 존재하는지 확인
    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()

In [103]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


C:\Users\KimHyeBin\AppData\Local\Temp\ipykernel_33596\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)
C:\Users\KimHyeBin\AppData\Local\Temp\ipykernel_33596\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and s

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [104]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [105]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [106]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [125]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

missing_tables = []
for name, df in raw_data.items():
    summary = missing_summary(df).reset_index()
    summary = summary.rename(columns={'index': 'column'})
    summary.insert(0, 'dataset', name)
    missing_tables.append(summary)

missing_table = pd.concat(missing_tables, ignore_index=True)

display(missing_table)

,dataset,column,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


과제 1 결과 해석: 원본 데이터셋의 모든 컬럼에서 결측치가 발견되지 않았다. 다만 데이터 타입 변환 과정에서 새로운 결측치가 생성될 수 있으므로 전처리 이후에도 재검증이 필요하다.

In [124]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

invalid_age = customers_clean[(customers_clean['age'] < 18) |(customers_clean['age'] > 100)]

print('범위 밖 나이 개수:', len(invalid_age))
print('최소 나이:', customers_clean['age'].min())
print('최대 나이:', customers_clean['age'].max())

display(invalid_age)

범위 밖 나이 개수: 0
최소 나이: 19
최대 나이: 69


,customer_id,name,gender,age,city,signup_date


과제 2 결과 해석: 고객 나이는 최소 19세, 최대 69세이며, 설정한 범위를 벗어난 값은 없다. 다만 18~100세는 이번 검증에서 사용한 기준이지, 모든 고객 데이터에 적용되는 절대적인 유효성 기준은 아니다.

In [123]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

monthly_orders = (orders_clean.groupby('order_month').size().reset_index(name='order_count').sort_values('order_month'))

display(monthly_orders)

,order_month,order_count
0,2025-09,9
1,2025-10,27
2,2025-11,22
3,2025-12,33
4,2026-01,29
5,2026-02,17
6,2026-03,35
7,2026-04,27
8,2026-05,28
9,2026-06,25


과제 3 결과 해석: 2026년 3월의 주문 건수가 35건으로 가장 많고, 2026년 9월이 8건으로 가장 적다. 다만 데이터 수집 기간이 월별로 동일한지 확인되지 않았기 때문에 단순히 주문이 감소했다고 판단할 수는 없다.
또한 이 결과는 취소 여부에 관계없이 모든 주문 상태를 포함한다.

In [126]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
top10_items = (order_items_clean.sort_values('line_total', ascending=False).head(10))

display(top10_items)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
493,494,190,70,5,198000,990000
78,79,35,70,5,198000,990000
631,632,247,58,5,197000,985000
604,605,235,58,5,197000,985000
113,114,51,58,5,197000,985000
569,570,221,58,5,197000,985000
45,46,21,58,5,197000,985000


과제 4 결과 해석: 주문 상세 항목의 최대 금액은 1,000,000원이다. 동일한 금액을 가진 항목이 여러 건 존재하며, 수량과 판매 단가가 모두 높은 항목에서 큰 금액이 나타났다.

## 1. 전처리 전 상태 기록
원본 데이터의 행 수, 컬럼 수, 결측치, 중복 행 수를 확인

In [111]:
# 전처리 전 원본 데이터 상태 기록
raw_summary = []

for name, df in raw_data.items():
    raw_summary.append({
        'dataset': name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_count': df.isna().sum().sum(),
        'duplicate_rows': df.duplicated().sum()
    })

raw_summary = pd.DataFrame(raw_summary)

print('===== 1. 전처리 전 상태 =====')
display(raw_summary)

===== 1. 전처리 전 상태 =====


,dataset,rows,columns,missing_count,duplicate_rows
0,customers,150,6,0,0
1,products,100,4,0,0
2,orders,300,5,0,0
3,order_items,764,5,0,0


## 2. 문자열·타입·결측 처리
전처리된 데이터의 컬럼 타입과 결측치 현황을 확인

In [112]:
# 전처리 후 데이터 타입과 결측치 확인
print('===== 2. 문자열·타입·결측 처리 =====')

for name, df in processed_data.items():
    print(f'\n[{name}]')

    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_ratio(%)': (df.isna().mean() * 100).round(2)
    })

    display(summary)

# 문자열 공백 처리 결과 확인
print('===== 문자열 처리 결과 =====')
display(customers_clean['city'].value_counts().head(10))

===== 2. 문자열·타입·결측 처리 =====

[customers]


,dtype,missing_count,missing_ratio(%)
customer_id,int64,0,0.0
name,str,0,0.0
gender,str,0,0.0
age,int64,0,0.0
city,str,0,0.0
signup_date,datetime64[us],0,0.0



[products]


,dtype,missing_count,missing_ratio(%)
product_id,int64,0,0.0
product_name,str,0,0.0
category,str,0,0.0
price,int64,0,0.0



[orders]


,dtype,missing_count,missing_ratio(%)
order_id,int64,0,0.0
customer_id,int64,0,0.0
order_date,datetime64[us],0,0.0
payment_method,str,0,0.0
order_status,str,0,0.0
order_month,str,0,0.0
order_dayofweek,str,0,0.0



[order_items]


,dtype,missing_count,missing_ratio(%)
order_item_id,int64,0,0.0
order_id,int64,0,0.0
product_id,int64,0,0.0
quantity,int64,0,0.0
unit_price,int64,0,0.0
line_total,int64,0,0.0


===== 문자열 처리 결과 =====


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

## 3. 중복·범주 표준화·이상값 후보
중복 개수, 주문 상태 종류, 가격·수량·단가의 이상값 후보

In [113]:
print('===== 3. 중복·범주 표준화·이상값 후보 =====')

# 전체 행 중복 확인
for name, df in processed_data.items():
    print(f'{name} 전체 행 중복: {df.duplicated().sum()}')

# 주문 상태 표준화 결과
print('\n[주문 상태 분포]')
display(orders_clean['order_status'].value_counts())

# 이상값 후보 확인
outlier_summary = pd.DataFrame({
    'column': ['age', 'price', 'quantity', 'unit_price'],
    'invalid_count': [
        ((customers_clean['age'] < 18) |
         (customers_clean['age'] > 100)).sum(),

        (products_clean['price'] <= 0).sum(),

        (order_items_clean['quantity'] <= 0).sum(),

        (order_items_clean['unit_price'] <= 0).sum()
    ]
})

print('\n[이상값 후보]')
display(outlier_summary)

===== 3. 중복·범주 표준화·이상값 후보 =====
customers 전체 행 중복: 0
products 전체 행 중복: 0
orders 전체 행 중복: 0
order_items 전체 행 중복: 0

[주문 상태 분포]


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


[이상값 후보]


,column,invalid_count
0,age,0
1,price,0
2,quantity,0
3,unit_price,0


## 4. PK/FK와 파생 컬럼 재검증
기본키 중복, 외래키 참조 관계, line_total 계산이 올바른지 확인

In [114]:
print('===== 4. PK/FK와 파생 컬럼 검증 =====')

# PK 후보 컬럼별 중복 및 결측 확인
key_columns = {
    'customers': 'customer_id',
    'products': 'product_id',
    'orders': 'order_id',
    'order_items': 'order_item_id'
}

pk_results = []

for name, key in key_columns.items():
    df = processed_data[name]

    pk_results.append({
        'dataset': name,
        'key': key,
        'duplicate_count': df[key].duplicated().sum(),
        'missing_count': df[key].isna().sum()
    })

print('\n[PK 후보 검증]')
display(pd.DataFrame(pk_results))

# FK 참조 관계 검증 (Notebook에서 생성한 결과)
print('\n[FK 검증]')
display(relationship_checks)

# line_total 계산 검증
line_total_check = (
    order_items_clean['line_total']
    == order_items_clean['quantity'] * order_items_clean['unit_price']
)

print('\n[line_total 검증]')
print('전체 행 수:', len(line_total_check))
print('계산 일치:', line_total_check.sum())
print('계산 불일치:', (~line_total_check).sum())

===== 4. PK/FK와 파생 컬럼 검증 =====

[PK 후보 검증]


,dataset,key,duplicate_count,missing_count
0,customers,customer_id,0,0
1,products,product_id,0,0
2,orders,order_id,0,0
3,order_items,order_item_id,0,0



[FK 검증]


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0



[line_total 검증]
전체 행 수: 764
계산 일치: 764
계산 불일치: 0



## 5. 전처리 전/후 비교
행 수와 컬럼 수의 변화 확인

In [115]:
print('===== 5. 전처리 전/후 비교 =====')

# comparison: Notebook에서 생성한 전처리 전후 비교표
display(comparison)

# 데이터셋별 행 수 감소 확인
for name in raw_data:
    before = len(raw_data[name])
    after = len(processed_data[name])

    print(f'{name}: {before} → {after} (감소: {before - after}행)')

===== 5. 전처리 전/후 비교 =====


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


customers: 150 → 150 (감소: 0행)
products: 100 → 100 (감소: 0행)
orders: 300 → 300 (감소: 0행)
order_items: 764 → 764 (감소: 0행)


## 6. 재현 가능한 전처리 확인
같은 원본 데이터를 전처리 함수에 두 번 입력했을 때 동일한 결과가 생성되는지 확인

In [116]:
print('===== 6. 전처리 재현성 검증 =====')

# Notebook에서 정의한 전처리 함수 사용
preprocess_functions = {
    'customers': preprocess_customers,
    'products': preprocess_products,
    'orders': preprocess_orders,
    'order_items': preprocess_order_items
}

results = []

for name, func in preprocess_functions.items():

    # 동일한 원본 데이터로 전처리 두 번 실행
    result1 = func(raw_data[name])
    result2 = func(raw_data[name])

    # equals(): 데이터프레임의 값, 인덱스, 컬럼, 타입 비교
    is_same = result1.equals(result2)

    results.append({
        'dataset': name,
        'first_run_rows': len(result1),
        'second_run_rows': len(result2),
        'reproducible': is_same
    })

display(pd.DataFrame(results))

===== 6. 전처리 재현성 검증 =====


C:\Users\KimHyeBin\AppData\Local\Temp\ipykernel_33596\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)
C:\Users\KimHyeBin\AppData\Local\Temp\ipykernel_33596\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and s

,dataset,first_run_rows,second_run_rows,reproducible
0,customers,150,150,True
1,products,100,100,True
2,orders,300,300,True
3,order_items,764,764,True



## 0. 제출 정보
- 이름: 김혜빈
- GitHub ID: Chloe-Hyebin-Kim
- 작성일: 2026년 10월 13일
- 최종 제출 URL: https://github.com/Chloe-Hyebin-Kim/llm-data-analysis-course/blob/main/practice/chapter05/chapter05.ipynb

## 1. 전처리 전 상태 기록
- 각 데이터 shape:
- 결측치:
- 전체 중복:
- 주요 ID 중복:
- 타입 문제 후보:

![전처리 전 상태](images/step01_before.png)

### 결과 관찰

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지와 이유를 작성하세요.

### 업무·분석적 의미

### 한계와 추가 확인 사항

## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리:
- 숫자 변환 대상과 실패 건수:
- 날짜 변환 대상과 실패 건수:
- 결측 처리 기준:

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.

### 업무·분석적 의미

### 한계와 추가 확인 사항

## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복:
- 표준화한 범주값:
- 허용값 밖 값:
- 이상값 후보:

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰

### 나의 해석과 판단
이상값을 자동 삭제하지 않고 후보로 검토해야 하는 이유를 작성하세요.

### 업무·분석적 의미

### 한계와 추가 확인 사항

## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`:
- 없는 `order_id`:
- 없는 `product_id`:
- 생성한 파생 컬럼:
- `line_total` 검증:

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰

### 나의 해석과 판단
전처리 후 관계를 다시 검증해야 하는 이유를 작성하세요.

## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | | | |
| 결측 | | | |
| 중복 | | | |
| 변환 실패 | | | |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰

### 나의 해석과 판단
행 수나 값 분포 변화가 의도한 결과인지 판단한 근거를 작성하세요.

### 업무·분석적 의미

### 한계와 추가 확인 사항

## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV:
- 생성된 요약 보고서:
- `python scripts/preprocess_data.py` 재실행 결과:
- 재실행 후 달라진 점:

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것의 차이를 작성하세요.

## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1.
2.
3.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

### 다음 EDA에서 특히 주의할 데이터 특성

### 현재 전처리의 한계

## 최종 제출 체크
- [ ] raw 원본을 덮어쓰지 않았습니다.
- [ ] 처리 전/후 비교를 남겼습니다.
- [ ] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [ ] clean CSV와 재실행 결과를 확인했습니다.
- [ ] 개인정보/Secret이 없습니다.
- [ ] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.
